# Proyecto Final — Airbnb: Madrid, Mallorca, Valencia, Sevilla, Milán
## Notebook 3: Análisis Estadístico

**Fuentes de datos:** Inside Airbnb (anuncios Airbnb) + Eurostat `tour_occ_nin3` (pernoctaciones turísticas)  
**Pregunta principal:** ¿Qué variables se asocian con mayor fuerza al precio por noche en Airbnb?

**Hipótesis a contrastar:**
1. Los precios difieren significativamente entre ciudades
2. Los superhosts presentan precios diferentes a los no-superhosts
3. La disponibilidad se asocia con el nivel de precio
4. Los anuncios populares presentan mejor valoración
5. La capacidad del alojamiento es el factor más correlacionado con el precio

> **Nota sobre el tamaño muestral:** Con 66.411 registros, prácticamente cualquier test dará p < 0.05 aunque la diferencia sea mínima. Por eso, además del p-valor, reportamos siempre la **magnitud del efecto** (diferencia de medianas, incremento porcentual, rho de Spearman). Significación estadística ≠ relevancia práctica.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)

SPAIN  = ['Madrid', 'Mallorca', 'Valencia', 'Sevilla']

df = pd.read_csv('../data/processed/listings_final.csv', low_memory=False)
df_spain = df[df['city'].isin(SPAIN)].copy()
CITIES = sorted(df['city'].unique().tolist())

print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Ciudades: {CITIES}')

## 1. Test de normalidad — Shapiro-Wilk

In [ ]:
sample = df['price'].sample(5000, random_state=42)
stat_sw, p_sw = stats.shapiro(sample)

print('TEST DE NORMALIDAD — Shapiro-Wilk (muestra n=5.000)')
print(f'H₀: price sigue distribución normal')
print(f'Estadístico W: {stat_sw:.4f}')
print(f'p-valor:       {p_sw:.2e}')
print(f'Asimetría:     {df["price"].skew():.3f}  (>1 indica fuerte sesgo derecha)')
print(f'Curtosis:      {df["price"].kurt():.3f}')
print()
print('→ Rechazamos H₀: price NO sigue distribución normal.')
print('→ Usaremos tests NO PARAMÉTRICOS (Kruskal-Wallis, Mann-Whitney, Spearman).')

## 2. H1: ¿Los precios difieren entre ciudades? — Kruskal-Wallis

In [ ]:
grupos = [df[df['city'] == city]['price'].values for city in CITIES]
stat_kw, p_kw = stats.kruskal(*grupos)

print('TEST KRUSKAL-WALLIS')
print('H₀: Los precios son iguales en todas las ciudades')
print(f'Estadístico H: {stat_kw:.2f} | p-valor: {p_kw:.2e}')
print()
medianas = df.groupby('city')['price'].median().sort_values(ascending=False)
print('Medianas por ciudad:')
print(medianas.round(1))
rango = medianas.max() - medianas.min()
print(f'\n→ Rechazamos H₀: diferencias estadísticamente significativas (p < 0.001).')
print(f'→ Magnitud: rango de medianas = {rango:.0f}€ '
      f'(+{rango/medianas.min()*100:.0f}% entre la más cara y la más barata).')

In [ ]:
# Post-hoc: comparaciones por pares
print('Comparaciones por pares (Mann-Whitney, bilateral):')
print(f'{"Par":<28} {"Med.A":>7} {"Med.B":>7} {"Dif.%":>8} {"p-valor":>12} {"Sig":>5}')
print('-' * 70)
for c1, c2 in combinations(CITIES, 2):
    g1 = df[df['city'] == c1]['price']
    g2 = df[df['city'] == c2]['price']
    _, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    dif_pct = (g1.median() - g2.median()) / g2.median() * 100
    sig = '✅' if p < 0.05 else '❌'
    print(f'{c1} vs {c2:<15} {g1.median():>7.1f} {g2.median():>7.1f} {dif_pct:>+8.1f}% {p:>12.2e} {sig:>5}')

> **Conclusión H1:** ✅ Confirmada. Diferencias estadísticamente significativas (p < 0.001) y de gran magnitud económica. Mallorca es el mercado más caro (+139% sobre Valencia). Las ciudades urbanas presentan medianas más homogéneas entre sí.

## 3. H2: Superhosts y precio — Mann-Whitney

> **Hipótesis revisada:** Dada la composición del dataset, no se asume a priori dirección en la diferencia. Se usa test bilateral para detectar cualquier diferencia significativa.

In [ ]:
sh_si = df[df['host_is_superhost'] == True]['price']
sh_no = df[df['host_is_superhost'] == False]['price']
_, p_mw1 = stats.mannwhitneyu(sh_si, sh_no, alternative='two-sided')

dif_abs = sh_si.median() - sh_no.median()
dif_pct = dif_abs / sh_no.median() * 100

print('TEST MANN-WHITNEY — Superhost vs precio (bilateral)\n')
print(f'  n superhost:     {len(sh_si):,}  |  mediana: {sh_si.median():.1f}€  |  media: {sh_si.mean():.1f}€')
print(f'  n no-superhost:  {len(sh_no):,}  |  mediana: {sh_no.median():.1f}€  |  media: {sh_no.mean():.1f}€')
print(f'  Diferencia mediana: {dif_abs:+.1f}€ ({dif_pct:+.1f}%)')
print(f'  p-valor: {p_mw1:.4f}')
print()
print('→ La diferencia global es estadísticamente significativa pero con efecto INVERSO al esperado:')
print('  los superhosts tienen mediana MENOR que los no-superhosts a nivel global.')
print()
print('→ Este resultado se explica por el efecto de composición: Mallorca (precios muy altos)')
print('  tiene un porcentaje de superhosts menor que el resto de ciudades.')
print()
print('Mediana precio por ciudad y superhost:')
print(df.groupby(['city','host_is_superhost'])['price'].median().unstack().round(1))
print()
print('→ Dentro de cada ciudad, los superhosts presentan precios IGUALES o SUPERIORES.')
print('  La diferencia global es un artefacto de la composición del dataset.')

> **Conclusión H2:** Resultado matizado. La diferencia global es significativa pero en sentido inverso al esperado, debido al efecto de composición entre ciudades (Mallorca tiene muchos anuncios caros pero pocos superhosts). **Dentro de cada ciudad**, los superhosts presentan precios iguales o superiores a los no-superhosts. Este es un ejemplo de la **paradoja de Simpson**: la tendencia a nivel global es contraria a la tendencia dentro de cada subgrupo.

## 4. H3: Disponibilidad y precio — Mann-Whitney

In [ ]:
alta = df[df['high_availability'] == True]['price']
baja = df[df['high_availability'] == False]['price']
_, p_mw2 = stats.mannwhitneyu(alta, baja, alternative='two-sided')

print('TEST MANN-WHITNEY — Disponibilidad vs precio (bilateral)\n')
print(f'  Alta disp. (>180d): n={len(alta):,}  |  mediana={alta.median():.1f}€')
print(f'  Baja disp. (≤180d): n={len(baja):,}  |  mediana={baja.median():.1f}€')
print(f'  Diferencia mediana: {alta.median()-baja.median():+.1f}€')
print(f'  p-valor: {p_mw2:.4f}')
print()
if p_mw2 > 0.05:
    print('→ No rechazamos H₀: no hay diferencia significativa en precio según disponibilidad.')
    print('→ Esto sugiere que el umbral de 180 días no separa bien grupos de precio diferente.')
    print('→ La disponibilidad tiene correlación casi nula con el precio (rho = +0.001).')
else:
    print('→ Diferencia significativa detectada.')

> **Conclusión H3:** ❌ No confirmada. No existe diferencia significativa en precio según el umbral de alta/baja disponibilidad. La correlación de Spearman entre `availability_365` y `price` es prácticamente nula (rho = +0.001). La disponibilidad en este dataset no es un predictor del precio.

## 5. H4: Popularidad y valoración — Mann-Whitney

In [ ]:
pop_si = df[df['is_popular'] == True]['review_scores_rating']
pop_no = df[df['is_popular'] == False]['review_scores_rating']
_, p_mw3 = stats.mannwhitneyu(pop_si, pop_no, alternative='greater')

print('TEST MANN-WHITNEY — Popularidad vs valoración\n')
print(f'  Populares:     n={len(pop_si):,}  |  mediana={pop_si.median():.4f}  |  media={pop_si.mean():.4f}')
print(f'  No populares:  n={len(pop_no):,}  |  mediana={pop_no.median():.4f}  |  media={pop_no.mean():.4f}')
print(f'  Diferencia mediana: {pop_si.median()-pop_no.median():+.4f} puntos')
print(f'  p-valor: {p_mw3:.4f}')
print()
if p_mw3 > 0.05:
    print('→ No hay diferencia significativa en valoración según popularidad.')
    print('→ Las medianas son idénticas (4.79), lo que refleja que la mayoría de')
    print('  alojamientos tienen valoraciones muy altas independientemente del volumen de reseñas.')
else:
    print('→ Diferencia significativa detectada.')

> **Conclusión H4:** ❌ No confirmada. Las medianas de valoración son idénticas (4.79) entre anuncios populares y no populares. El mercado Airbnb presenta un fuerte efecto de selección positiva en valoraciones: los alojamientos con malas reseñas desaparecen del mercado, por lo que la distribución de valoraciones está muy comprimida en el rango 4.5-5.0 y no discrimina entre anuncios.

## 6. H5: Correlaciones con el precio — Spearman

In [ ]:
cols_corr = [
    'accommodates','bathrooms','bedrooms','beds',
    'minimum_nights','availability_365','number_of_reviews',
    'review_scores_rating','review_scores_location',
    'reviews_per_month','calculated_host_listings_count'
]

resultados = []
for var in cols_corr:
    rho, p = stats.spearmanr(df['price'], df[var])
    resultados.append({
        'Variable': var, 'Rho Spearman': round(rho,4),
        'p-valor': p, 'Sig': '✅' if p < 0.05 else '❌'
    })

corr_df = pd.DataFrame(resultados).sort_values('Rho Spearman', key=abs, ascending=False)
print('Correlaciones de Spearman con price:')
corr_df

In [ ]:
# Turismo vs precio — solo España (comparabilidad territorial)
rho_t, p_t   = stats.spearmanr(df_spain['pernoctaciones_total'], df_spain['price'])
rho_tp, p_tp = stats.spearmanr(df_spain['tourism_pressure'], df_spain['price'])

print('Correlación turismo (Eurostat) vs precio — ciudades españolas:')
print(f'  pernoctaciones_total:  rho={rho_t:.4f}  p={p_t:.2e}')
print(f'  tourism_pressure:      rho={rho_tp:.4f}  p={p_tp:.2e}')
print()
print('→ Correlación positiva moderada (rho~0.38) entre presión turística y precio en España.')
print('  Milán se excluye para mantener comparabilidad territorial.')

> **Conclusión H5:** ✅ Confirmada. `accommodates` (rho=0.61) es la variable más correlacionada con el precio, seguida de `bedrooms` (0.53), `beds` (0.52) y `bathrooms` (0.45). Las variables de reputación tienen asociación débil o negativa. La presión turística muestra correlación positiva moderada en España (rho=0.38).

## 7. Modelo explicativo del precio — Regresión Lineal

Construimos un modelo de regresión lineal para **cuantificar el peso de cada variable en la predicción del precio**. El objetivo es explicativo, no predictivo: nos permite comparar la contribución de cada variable controlando el resto.

In [ ]:
cols_modelo = [
    'price','accommodates','bathrooms','bedrooms','beds',
    'availability_365','number_of_reviews','review_scores_rating',
    'calculated_host_listings_count'
]

df_model = df[cols_modelo].dropna()
print(f'Registros para el modelo: {len(df_model):,}')

X = df_model.drop(columns='price')
y = df_model['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

r2  = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

print(f'\nMétricas del modelo:')
print(f'  R²:  {r2:.4f} → el modelo explica el {r2*100:.1f}% de la varianza del precio')
print(f'  MAE: {mae:.2f}€  → error medio absoluto de predicción')

> **Sobre el modelo:** El modelo de regresión lineal presenta un R² de 0.39, lo que indica que las variables numéricas incluidas explican aproximadamente el 39% de la variabilidad del precio. El porcentaje restante corresponde a factores cualitativos no recogidos en el dataset: ubicación exacta del alojamiento dentro del barrio, calidad de las fotografías, políticas de cancelación, temporada de reserva y características decorativas.
>
> Esto es **esperable en un mercado complejo como Airbnb**, donde intervienen factores no observados como la ubicación exacta dentro del barrio, la calidad de las fotografías, la temporada de reserva o las características decorativas del alojamiento.

In [ ]:
# Coeficientes brutos
coef_df = pd.DataFrame({
    'variable': X.columns,
    'coeficiente_€': model.coef_
}).sort_values('coeficiente_€', ascending=False)

print('Coeficientes del modelo (€ de cambio en precio por unidad de cada variable):')
print(coef_df.round(3).to_string(index=False))

In [ ]:
# Coeficientes estandarizados — importancia relativa
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model_std = LinearRegression()
model_std.fit(X_scaled, y)

coef_std = pd.DataFrame({
    'variable': X.columns,
    'coef_estandarizado': model_std.coef_
}).sort_values('coef_estandarizado', key=abs, ascending=False)

print('Coeficientes estandarizados — importancia relativa entre variables:')
print(coef_std.round(3).to_string(index=False))

> **Interpretación del modelo:**
> - **Variables estructurales dominan**: `accommodates` (coef. std. 57.1), `bathrooms` (49.1) y `bedrooms` (17.2) tienen los coeficientes estandarizados más altos. Son los predictores más potentes.
> - **`bathrooms` tiene el mayor coeficiente bruto** (+54€ por baño adicional): un baño extra añade más precio que una persona más de capacidad, lo que refleja que los alojamientos grandes y con múltiples baños son los más caros.
> - **`number_of_reviews` y `beds` tienen coeficiente negativo**: los alojamientos con más reseñas son los más baratos (mayor rotación). Más camas sin más baños o habitaciones indica alojamientos menos exclusivos.
> - **`review_scores_rating` tiene efecto positivo pero pequeño** (coef. std. 4.7): la valoración importa pero es secundaria frente al tamaño.

## 8. Resumen estadístico

| Hipótesis | Test | p-valor | Magnitud | Resultado |
|---|---|---|---|---|
| H1: Precios difieren entre ciudades | Kruskal-Wallis | <0.001 | Rango mediana 100€–239€ (+139%) | ✅ Confirmada |
| H2: Superhosts → precios diferentes | Mann-Whitney | <0.001 | Efecto global inverso (paradoja Simpson) | ⚠️ Matizada |
| H3: Disponibilidad → precio | Mann-Whitney | 0.74 | Sin diferencia significativa | ❌ Rechazada |
| H4: Populares → mejor valoración | Mann-Whitney | 1.00 | Medianas idénticas (4.79) | ❌ Rechazada |
| H5: Capacidad = mayor correlación | Spearman | <0.001 | rho = 0.61 en accommodates | ✅ Confirmada |

---

**Conclusión del análisis estadístico:** Las variables estructurales del alojamiento — capacidad (`accommodates`), número de dormitorios (`bedrooms`) y número de baños (`bathrooms`) — son las que mejor explican el precio por noche, tanto en correlaciones de Spearman como en el modelo de regresión lineal (R² = 0.39). Las variables de reputación (valoración, reseñas) tienen un efecto estadísticamente secundario. Los resultados sobre superhosts y disponibilidad ilustran la importancia de analizar la magnitud del efecto y el efecto de composición del dataset, no solo la significación estadística.

> **Respuesta directa a la pregunta del proyecto:** Las variables que mejor explican el precio por noche en Airbnb son la **capacidad del alojamiento** (`accommodates`), el **número de baños** (`bathrooms`) y el **número de dormitorios** (`bedrooms`). Las reseñas y la disponibilidad tienen un efecto muy limitado. La ciudad actúa como variable de contexto que amplifica o modera estos efectos de forma significativa.

**En conjunto:** el precio de los alojamientos Airbnb está principalmente determinado por las **características estructurales del alojamiento** (capacidad, número de dormitorios, número de baños), mientras que los factores reputacionales (valoración, número de reseñas) tienen un impacto limitado y secundario.

> El modelo presenta un **R² de 0.39**, lo que indica que las variables incluidas explican parcialmente la variabilidad del precio. El porcentaje restante refleja factores no observados como la ubicación exacta, la temporada o las características cualitativas del alojamiento.